# Differentiable Monte Carlo — The Real Deal

**Goal:** Convert the paper's MC simulation into a fully differentiable pipeline in PyTorch.

---

## Step 1: Imports & Setup

In [1]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import dataclass
import math

## Step 2: Parameters & Sampling Functions

In [2]:
@dataclass
class MCParams:
    """Parameters of the Monte Carlo PLE forward model.

    Forward model per run: draw ~N photons whose detunings follow a Lorentzian
    (Cauchy) line of half-width `gamma`, plus a uniform background.

    Attributes:
        gamma:   Lorentzian HWHM, in MHz. **Optimized** — the quantity we
                 reconstruct (line FWHM = 2 * gamma).
        nbar:    Mean photon count per run. **Optimized**.
        sigma:   Std of the photon-count noise, N(nbar, sigma). Fixed (paper).
        lambda_: Mean number of uniform background counts per run. Fixed (paper).
    """
    gamma: float
    nbar: float
    sigma: float = 6.0
    lambda_: float = 2.0


# Detuning window the spectrometer records, in MHz (from paper).
FREQ_MIN = -75.0
FREQ_MAX = 75.0


def sample_n(params, epsilon):
    """Draw the photon count for one run, N ~ Normal(nbar, sigma).

    Reparameterized: the randomness is the external standard-normal `epsilon`,
    so N is a deterministic function of the params given `epsilon`. Rounded to a
    non-negative integer.

    Args:
        params:  MCParams — uses `nbar` (mean) and `sigma` (std).
        epsilon: float — a standard-normal draw, N(0, 1).

    Returns:
        int — photon count for this run, >= 0.

    Note:
        `round` is non-differentiable, so gradients do not yet flow to `nbar`
        through the count. Making this differentiable is part of the TODO.
    """
    n_float = params.nbar + params.sigma * epsilon
    return max(round(n_float), 0)


def sample_cauchy(n, gamma):
    """Draw `n` photon detunings from a Lorentzian (Cauchy) line centered at 0.

    Inverse-transform sampling: for u ~ Uniform(0, 1),
        f = gamma * tan(pi * (u - 0.5))
    is Cauchy(0, gamma), a Lorentzian of HWHM `gamma`. Samples are clipped to the
    recorded window [FREQ_MIN, FREQ_MAX] (Cauchy tails are heavy and would
    otherwise produce arbitrarily large detunings).

    Args:
        n:     int — number of photons to draw.
        gamma: float — Lorentzian HWHM, in MHz.

    Returns:
        (n,) float32 tensor of detunings (MHz); empty if n == 0.

    Note:
        Sampling runs in NumPy and is wrapped in a fresh tensor, so this is NOT
        differentiable in `gamma` yet. Reparameterizing it (draw `u` in torch,
        keep `gamma` a tensor) is the key step for the differentiable pipeline.
    """
    if n == 0:
        return torch.tensor([], dtype=torch.float32)
    u = np.random.uniform(0, 1, n)
    samples = gamma * np.tan(np.pi * (u - 0.5))
    return torch.tensor(np.clip(samples, FREQ_MIN, FREQ_MAX), dtype=torch.float32)


def sample_background(lambda_):
    """Draw background counts: Poisson(lambda_) events, uniform over the window.

    Args:
        lambda_: float — mean number of background counts per run.

    Returns:
        (n_bg,) float32 tensor of uniform detunings in [FREQ_MIN, FREQ_MAX];
        empty if zero background events were drawn.
    """
    n_bg = np.random.poisson(lambda_)
    if n_bg == 0:
        return torch.tensor([], dtype=torch.float32)
    bg = np.random.uniform(FREQ_MIN, FREQ_MAX, n_bg)
    return torch.tensor(bg, dtype=torch.float32)


def sample_photons(n, gamma, lambda_):
    """Build one run's photon set: Lorentzian signal + uniform background.

    Args:
        n:       int — number of signal photons.
        gamma:   float — Lorentzian HWHM (MHz).
        lambda_: float — mean background counts.

    Returns:
        (n + n_bg,) float32 tensor of all detunings (MHz) for this run.
    """
    signal = sample_cauchy(n, gamma)
    bg = sample_background(lambda_)
    return torch.cat([signal, bg])

## Step 3: Fitting — Voigt MLE

In [3]:
def pseudo_voigt_log_pdf(freqs, center, log_gamma, log_sigma_g, logit_eta):
    """Log-density of a normalized pseudo-Voigt at each frequency.

    A pseudo-Voigt is a convex mix of a Gaussian and a Lorentzian sharing a
    center: pdf = eta * Gaussian + (1 - eta) * Lorentzian. Shape parameters are
    passed in *unconstrained* form and mapped to their valid ranges inside, so an
    unconstrained optimizer (L-BFGS) can search freely:
        gamma   = exp(log_gamma)        > 0       (Lorentzian HWHM)
        sigma_g = exp(log_sigma_g)      > 0       (Gaussian std)
        eta     = sigmoid(logit_eta)    in (0, 1) (mixing weight)

    Args:
        freqs:       (N,) tensor — photon detunings (MHz).
        center:      scalar tensor — line center (MHz).
        log_gamma:   scalar tensor — log of the Lorentzian HWHM.
        log_sigma_g: scalar tensor — log of the Gaussian std.
        logit_eta:   scalar tensor — logit of the Gaussian/Lorentzian mix weight.

    Returns:
        (N,) tensor — log pdf at each frequency. A 1e-30 floor inside the log
        guards against log(0).
    """
    gamma = torch.exp(log_gamma)
    sigma_g = torch.exp(log_sigma_g)
    eta = torch.sigmoid(logit_eta)

    gauss = torch.exp(-0.5 * ((freqs - center) / sigma_g) ** 2)
    gauss = gauss / (sigma_g * torch.sqrt(torch.tensor(2.0 * torch.pi)))

    lorentz = (gamma / torch.pi) / ((freqs - center) ** 2 + gamma ** 2)

    pdf = eta * gauss + (1 - eta) * lorentz
    return torch.log(pdf + 1e-30)

def fit_pseudo_voigt(photons, n_iters=200):
    """Fit a pseudo-Voigt to one run's photons by maximum likelihood (L-BFGS).

    Minimizes the mean negative log-likelihood of the detunings under
    `pseudo_voigt_log_pdf`, optimizing the center plus the three shape parameters
    in unconstrained space. The reported linewidth is FWHM = 2 * gamma.

    Args:
        photons: (N,) tensor — detunings (MHz) for one run.
        n_iters: int — max L-BFGS iterations.

    Returns:
        (fwhm, params):
            fwhm:   float — fitted linewidth (MHz), = 2 * gamma.
            params: dict (center, gamma, sigma_g, eta, fwhm), or None on failure.

    Failure handling:
        Returns the sentinel (50.0, None) when there are < 3 photons, the
        optimizer raises, or gamma comes out non-finite.
    """
    if len(photons) < 3:
        return 50.0, None

    freqs = photons.clone().detach().float()

    center = torch.tensor(float(freqs.median()), requires_grad=True)
    log_gamma = torch.tensor(np.log(15.0), requires_grad=True)
    log_sigma_g = torch.tensor(np.log(5.0), requires_grad=True)
    logit_eta = torch.tensor(0.0, requires_grad=True)

    optimizer = torch.optim.LBFGS([center, log_gamma, log_sigma_g, logit_eta],
                                   max_iter=n_iters, line_search_fn='strong_wolfe')

    def closure():
        optimizer.zero_grad()
        log_pdf = pseudo_voigt_log_pdf(freqs, center, log_gamma, log_sigma_g, logit_eta)
        nll = -log_pdf.mean()
        nll.backward()
        return nll

    try:
        optimizer.step(closure)
    except RuntimeError:
        return 50.0, None

    gamma_val = torch.exp(log_gamma).item()
    if not math.isfinite(gamma_val):
        return 50.0, None
    fwhm = 2.0 * gamma_val

    return fwhm, {
        'center': center.item(),
        'gamma': gamma_val,
        'sigma_g': torch.exp(log_sigma_g).item(),
        'eta': torch.sigmoid(logit_eta).item(),
        'fwhm': fwhm,
    }

## Step 4: Full Run — Params In, FWHM Out

In [4]:
def full_run(params, epsilon):
    """One Monte Carlo run end-to-end: params -> photons -> fit -> FWHM.

    Pipeline: draw the photon count N (via `epsilon`), sample Lorentzian signal +
    uniform background detunings, then fit a pseudo-Voigt and return its width.

    Args:
        params:  MCParams — model parameters for this run.
        epsilon: float — standard-normal draw feeding the photon-count sampler.

    Returns:
        float — fitted FWHM (MHz) for this run, or the sentinel 50.0 if fewer
        than 3 photons were produced.
    """
    n = sample_n(params, epsilon)
    photons = sample_photons(n, params.gamma, params.lambda_)
    if len(photons) < 3:
        return 50.0  # too few photons to fit — return safe default

    fwhm, _ = fit_pseudo_voigt(photons)
    return fwhm

## Step 5: Full Simulation

In [5]:
def simulate(params, n_runs=2000, seed=None):
    """Run the MC forward model many times -> empirical FWHM distribution.

    Repeats `full_run` `n_runs` times with the same parameters but independent
    noise, yielding one fitted FWHM per run. This collection of FWHMs is the
    simulated distribution we compare against the experimental one (via MMD²).

    Args:
        params: MCParams — model parameters (same for every run).
        n_runs: int — number of independent runs / FWHM samples to produce.
        seed:   int or None — NumPy seed for reproducibility.

    Returns:
        (n_runs,) tensor of fitted FWHMs (MHz).

    Note:
        A Python loop with an L-BFGS fit per run — the main cost of the pipeline.
        Not differentiable end-to-end yet (see the sampling/round caveats above).
    """
    if seed is not None:
        np.random.seed(seed)

    fwhms = torch.zeros(n_runs)
    for i in range(n_runs):
        eps = np.random.normal()
        fwhms[i] = full_run(params, eps)

    return fwhms

---
Functions loaded. Ready for testing.

## Step 6: Loss Function — Biased MMD²

The real data turned out to be **samples** of FWHM values, not a histogram. So
both sides are now sets of samples — simulated (`x`, depends on params) and real
(`y`, fixed) — and we can compare the two distributions **directly** with the
**Maximum Mean Discrepancy (MMD)**. No binning, no `kde_to_bin_counts` workaround.

`mmd2_biased(x, y)` computes the *biased* MMD² estimator with a Gaussian kernel:

$$\widehat{\mathrm{MMD}}^2_b = \frac{1}{m^2}\sum_{i,j} k(x_i,x_j)
  + \frac{1}{n^2}\sum_{i,j} k(y_i,y_j)
  - \frac{2}{mn}\sum_{i,j} k(x_i,y_j)$$

- Differentiable in `x` → gradients flow back to the simulation params.
- Handles **unequal sample sizes** (`m != n`) naturally.
- Bandwidth defaults to the **median heuristic** (detached → treated as a constant).

> **Note:** this data's FWHM spans several orders of magnitude. Consider passing
> `log`-transformed samples (e.g. `torch.log10(fwhms)`) so the single kernel scale
> is meaningful across the whole range.

In [6]:
def gaussian_gram(a, b, bandwidth):
    """Gaussian kernel (Gram) matrix between two 1-D sample sets.

    Computes k(a_i, b_j) = exp(-(a_i - b_j)^2 / (2 * bandwidth^2)) for all pairs
    via broadcasting (no Python loop) — vectorized and differentiable.

    Args:
        a:         (m,) tensor.
        b:         (n,) tensor.
        bandwidth: scalar — kernel length scale (same units as the samples).

    Returns:
        (m, n) tensor of pairwise kernel similarities in (0, 1].
    """
    d2 = (a[:, None] - b[None, :]) ** 2          # (m, n) squared distances
    return torch.exp(-d2 / (2.0 * bandwidth ** 2))


def median_bandwidth(x, y):
    """Median-heuristic kernel bandwidth from the pooled samples.

    Sets the Gaussian kernel scale to the median pairwise distance over the
    combined samples — a standard, parameter-free default for MMD. Detached, so
    it acts as a constant and carries no gradients.

    Args:
        x: (m,) tensor.
        y: (n,) tensor.

    Returns:
        scalar tensor — bandwidth (floored at 1e-12 to avoid divide-by-zero).
    """
    z = torch.cat([x, y]).detach()
    d2 = (z[:, None] - z[None, :]) ** 2
    iu = torch.triu_indices(len(z), len(z), offset=1)   # unique pairs (i < j)
    med = torch.median(d2[iu[0], iu[1]]).sqrt()
    return med.clamp_min(1e-12)


def mmd2_biased(x, y, bandwidth=None):
    """Biased MMD² between simulated samples `x` and real samples `y`.

    Maximum Mean Discrepancy with a Gaussian kernel, biased (V-statistic)
    estimator — the within-set means include the diagonal terms:

        MMD²_b = mean(Kxx) + mean(Kyy) - 2 * mean(Kxy)

    Zero iff the two empirical distributions match, differentiable in `x` (so
    gradients reach the simulation params), and handles unequal sizes.

    Args:
        x:         (m,) tensor — simulated FWHMs (carries gradients).
        y:         (n,) tensor — real FWHMs (fixed target).
        bandwidth: float or None — kernel scale; None -> median heuristic.

    Returns:
        scalar tensor, >= 0, differentiable in `x`.

    Note:
        Builds full (m, m), (n, n) and (m, n) kernel matrices, so memory grows as
        O(m² + n²) — subsample very large sets before calling.
    """
    if bandwidth is None:
        bandwidth = median_bandwidth(x, y)
    Kxx = gaussian_gram(x, x, bandwidth)
    Kyy = gaussian_gram(y, y, bandwidth)
    Kxy = gaussian_gram(x, y, bandwidth)
    # .mean() includes the diagonal -> biased V-statistic estimator
    return Kxx.mean() + Kyy.mean() - 2.0 * Kxy.mean()


# Test: real samples from "true" params, compare guesses via MMD^2
print("--- Testing MMD^2 loss ---")

true_params = MCParams(gamma=20.0, nbar=50.0)
real_fwhms = simulate(true_params, n_runs=500, seed=123)   # fixed target (samples)

guess_params = MCParams(gamma=10.0, nbar=40.0)
sim_fwhms = simulate(guess_params, n_runs=500, seed=456)
loss = mmd2_biased(sim_fwhms, real_fwhms)

print(f'Real:     gamma={true_params.gamma}, nbar={true_params.nbar}')
print(f'Guess:    gamma={guess_params.gamma}, nbar={guess_params.nbar}')
print(f'MMD^2:    {loss:.6f}')

# Same params should give a lower MMD^2 (different random draws -> small, not 0)
same_fwhms = simulate(MCParams(gamma=20.0, nbar=50.0), n_runs=500, seed=789)
same_loss = mmd2_biased(same_fwhms, real_fwhms)
print(f'Same params MMD^2: {same_loss:.6f} (should be lower)')

--- Testing MMD^2 loss ---


Real:     gamma=20.0, nbar=50.0
Guess:    gamma=10.0, nbar=40.0
MMD^2:    0.087773


Same params MMD^2: 0.003238 (should be lower)
